## ☕ AI Café Assistant
#### A fun chatbot that helps you explore a café menu, pick what to eat or drink, and discover tasty combos. Just chat naturally and get suggestions for coffee, cakes, muffins, and shakes.

#### Based on Google Gemini model and Gradio chat interface

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [3]:

load_dotenv(override=True)
gemini_api_key = os.getenv('GOOGLE_API_KEY')

if gemini_api_key:
    print(f"Gemini API Key exists and begins {gemini_api_key[:8]}")
else:
    print("Gemini API Key not set")

Gemini API Key exists and begins AIzaSyA3


In [4]:
# Initialize Gemini

gemini = OpenAI(
    api_key=gemini_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

MODEL = "gemini-2.5-flash"

In [5]:
system_message = """
You are a friendly and helpful cafe assistant.
You help customers choose from the Corner cafe menu and make gentle recommendations.

Main categories:
- Coffee
- Cakes
- Muffins
- Shakes

General rules:
- Always answer the customer's main request first
- If they ask about one category, use that category as the main focus
- After answering, gently recommend 1 or 2 items from other categories that pair well
- Keep responses warm, short, and natural
- If the customer is unsure, suggest popular items
- If the customer asks for something not sold in the cafe, politely say it is unavailable and guide them toward available items
"""

In [6]:
coffee_prompt = """
The customer is asking about coffee.
Focus mainly on coffee items such as Espresso, Cappuccino, Latte, Americano, and Iced Latte.
After helping with coffee, also suggest one matching item from cakes or muffins, and optionally a shake if it fits.
Examples:
- Coffee goes well with Chocolate Cake
- Cappuccino pairs nicely with a Blueberry Muffin
"""

cakes_prompt = """
The customer is asking about cakes.
Focus mainly on cake items such as Chocolate Cake, Cheesecake, Red Velvet Cake, and Black Forest Cake.
After helping with cakes, also suggest one matching coffee or shake.
Examples:
- Cheesecake goes well with a Latte
- Chocolate Cake pairs well with Cappuccino
"""

muffins_prompt = """
The customer is asking about muffins.
Focus mainly on muffin items such as Blueberry Muffin, Chocolate Muffin, Vanilla Muffin, and Banana Muffin.
After helping with muffins, also suggest one matching coffee or shake.
Examples:
- Blueberry Muffin pairs well with an Iced Latte
- Chocolate Muffin goes nicely with Cappuccino
"""

shakes_prompt = """
The customer is asking about shakes.
Focus mainly on shake items such as Vanilla Shake, Chocolate Shake, Strawberry Shake, and Mango Shake.
After helping with shakes, also suggest one light dessert item like cake or muffin.
Examples:
- Mango Shake goes well with a Vanilla Muffin
- Chocolate Shake pairs nicely with Chocolate Cake
"""

fallback_prompt = """
The customer has not clearly asked for one category.
Ask what they are in the mood for: Coffee, Cakes, Muffins, or Shakes.
You may also mention a few popular items from different categories.
"""


In [7]:

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    msg = message.lower()
    relevant_system_message = system_message

    if any(word in msg for word in ["coffee", "espresso", "latte", "cappuccino", "americano"]):
        relevant_system_message += "\n" + coffee_prompt

    elif any(word in msg for word in ["cake", "cakes", "cheesecake", "red velvet", "black forest"]):
        relevant_system_message += "\n" + cakes_prompt

    elif any(word in msg for word in ["muffin", "muffins"]):
        relevant_system_message += "\n" + muffins_prompt

    elif any(word in msg for word in ["shake", "shakes"]):
        relevant_system_message += "\n" + shakes_prompt

    else:
        relevant_system_message += "\n" + fallback_prompt
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = gemini.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [8]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.
